# INSTALLING ALL PACKAGES SECTION

%pip install numpy scipy astropy

%pip install certifi

%pip install pytest pandas

%pip install h5py

# CHOENO

# GETTING THE DATA 

In [1]:
import numpy as np
import h5py
from astropy.io import fits
import glob
import firstmap_b as fm

done in terminal

# RUNNING THE MAIN MAP FILES

In [2]:
%run first_map.py --fake

[check] uniform in-sky field: p = 0.15789 = p0/(1-p0/3)  OK

run: FAKE_meanB_along_x   shape: (64, 64, 64)
mean field:  <Bx> = +1.0000  <By> = +0.0000  <Bz> = +0.0000
rms field:   rms Bx = 0.3000  rms By = 0.3000  rms Bz = 0.3000

LOS   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS  g_obs(p0 known)
  z    0.0   13.1  0.1482  0.1412   2.16   21.3   15.2  +21.3   +15.2             20.3
  y    0.0   15.5  0.1499  0.1428   1.95   18.1   17.6  +18.1   +17.6             21.4
  x   90.0   67.4  0.0277  0.0275   9.19   56.3   46.5  -33.7   -43.5             75.5

appended 3 rows to results.csv


# Seeing whats in the data files

In [3]:
from astropy.io import fits
import numpy as np
for name in ["dens", "magx", "magy", "magz"]:
    with fits.open(f"b1p1/{name}_t900.fits.gz") as h:
        h.info()                          # what's inside the file
        d = h[0].data
        print(name, d.shape, d.dtype,
              f"mean={d.mean():.3f}  min={d.min():.3f}  max={d.max():.3f}")
        print(repr(h[0].header)[:800])    # header: may contain extra notes
        print()

Filename: b1p1/dens_t900.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      10   (256, 256, 256)   float32   
dens (256, 256, 256) >f4 mean=1.000  min=0.276  max=3.665
SIMPLE  =                    T / Written by IDL:  Wed Nov  8 11:53:39 2017      
BITPIX  =                  -32 / Number of bits per data pixel                  
NAXIS   =                    3 / Number of data axes                            
NAXIS1  =                  256 /                                                
NAXIS2  =                  256 /                                                
NAXIS3  =                  256 /                                                
EXTEND  =                    T / FITS data may contain extensions               
DATE    = '2017-11-08'         / Creation UTC (CCCC-MM-DD) date of FITS header  
COMMENT FITS (Flexible Image Transport System) format is defined in 'Astronomy  
COMMENT and Astrophysics', volume 376, page 359

In [4]:
B = {k: fits.getdata(f"b1p1/mag{k}_t900.fits.gz").astype(float) for k in "xyz"}

def div(bx_axis, by_axis, bz_axis):
    d = lambda f, a: (np.roll(f, -1, a) - np.roll(f, 1, a)) / 2
    return np.abs(d(B["x"], bx_axis) + d(B["y"], by_axis) + d(B["z"], bz_axis)).mean()

print("x=axis2, y=axis1, z=axis0 (script's assumption):", div(2, 1, 0))
print("x=axis0, y=axis1, z=axis2 (reversed):           ", div(0, 1, 2))

x=axis2, y=axis1, z=axis0 (script's assumption): 0.001651106264493735
x=axis0, y=axis1, z=axis2 (reversed):            0.03917176250175174


In [5]:
%run first_map.py --rho b1p1/dens_t900.fits.gz --bx b1p1/magx_t900.fits.gz --by b1p1/magy_t900.fits.gz --bz b1p1/magz_t900.fits.gz --run b1p1_t900

[check] uniform in-sky field: p = 0.15789 = p0/(1-p0/3)  OK

run: b1p1_t900   shape: (256, 256, 256)
mean field:  <Bx> = +1.0000  <By> = +0.0000  <Bz> = -0.0000
rms field:   rms Bx = 0.3102  rms By = 0.3380  rms Bz = 0.3553

LOS   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS  g_obs(p0 known)
  z    0.0   16.0  0.1411  0.1348   0.34   25.1   23.0  +25.1   +23.0             30.2
  y    0.0   17.8  0.1395  0.1333   0.35   25.7   24.2  +25.7   +24.2             30.9
  x   90.0   66.3  0.0507  0.0498   2.98   62.6   57.0  -27.4   -33.0             73.9

appended 3 rows to results.csv


In [6]:
%run rotate_maps.py --rho b1p1/dens_t900.fits.gz --bx b1p1/magx_t900.fits.gz --by b1p1/magy_t900.fits.gz --bz b1p1/magz_t900.fits.gz --run b1p1_t900

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: b1p1_t900   shape: (256, 256, 256)   p_max rule: 99.5th percentile
mean field: <Bx>=+1.0000 <By>=+0.0000 <Bz>=-0.0000  -> field along x; tilting toward z

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.0   16.1  0.1411  0.1348   0.34   25.0   22.9  +25.0   +22.9    30.1     28.5   3.1
   15   15.0   26.5  0.1311  0.1256   0.39   31.8   26.1  +16.8   +11.1    38.3     34.0   3.8
   30   30.0   29.3  0.1135  0.1093   0.50   32.3   28.9   +2.3    -1.1    41.7     40.4   5.5
   45   45.0   47.6  0.0967  0.0937   0.80   47.3   47.3   +2.3    +2.3    56.8     49.7   5.9
   60   60.0   53.6  0.0792  0.0771   1.56   54.5   52.7   -5.5    -7.3    64.6     63.6   6.9
   75   75.0   63.0  0.0601  0.0590   2.66   64.0   59.2  -11.0   -15.8    72.9     70.7   6.9
   90   90.0   67.2  0.0507  0.0498   2.

In [7]:
for run in ["b1p.01", "b.1p1", "b.1p.01"]:
    d = glob.glob(f"{run}/dens_*.fits.gz")[0]
    t = d.split("dens_")[1].replace(".fits.gz", "")
    %run rotate_maps.py --rho {run}/dens_{t}.fits.gz --bx {run}/magx_{t}.fits.gz --by {run}/magy_{t}.fits.gz --bz {run}/magz_{t}.fits.gz --run {run}_{t}

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: b1p.01_t900   shape: (256, 256, 256)   p_max rule: 99.5th percentile
mean field: <Bx>=+1.0000 <By>=-0.0000 <Bz>=+0.0000  -> field along x; tilting toward z

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.0   16.9  0.1432  0.1366   1.09   22.7   21.1  +22.7   +21.1    27.2     26.3   3.3
   15   15.0   20.1  0.1413  0.1349   1.08   26.1   20.5  +11.1    +5.5    27.7     26.7   3.8
   30   30.0   31.8  0.1352  0.1293   1.14   29.1   29.1   -0.9    -0.9    34.9     35.1   4.7
   45   45.0   41.7  0.1218  0.1171   1.46   41.8   40.6   -3.2    -4.4    48.0     47.3   5.6
   60   60.0   52.4  0.0920  0.0893   2.32   51.7   50.2   -8.3    -9.8    60.4     57.0   6.7
   75   75.0   61.9  0.0628  0.0615   3.95   60.5   54.4  -14.5   -20.6    70.4     67.2   6.3
   90   90.0   63.6  0.0636  0.0622   

IndexError: list index out of range

In [8]:
%run rotate_maps.py --rho b.1p.01/dens_t750.fits --bx b.1p.01/magx_t750.fits.gz --by b.1p.01/magy_t750.fits.gz --bz b.1p.01/magz_t750.fits.gz --run b.1p.01_t750

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: b.1p.01_t750   shape: (256, 256, 256)   p_max rule: 99.5th percentile
mean field: <Bx>=+0.1000 <By>=-0.0000 <Bz>=+0.0000  -> field along x; tilting toward z

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.0   31.4  0.0806  0.0785   6.27   55.7   48.0  +55.7   +48.0    62.0     59.6   3.2
   15   15.0   32.4  0.0878  0.0853   6.25   55.4   50.8  +40.4   +35.8    63.6     60.6   3.8
   30   30.0   32.3  0.0829  0.0807   5.88   53.1   46.7  +23.1   +16.7    62.8     58.8   4.4
   45   45.0   34.0  0.0845  0.0822   5.93   52.9   48.9   +7.9    +3.9    62.3     58.5   8.4
   60   60.0   34.1  0.0798  0.0777   6.21   55.4   46.2   -4.6   -13.8    63.8     59.1   8.4
   75   75.0   33.4  0.0858  0.0834   6.16   54.7   50.1  -20.3   -24.9    63.7     60.5   6.5
   90   90.0   33.1  0.0800  0.0780  

In [9]:
%run evaluate.py

         run  estimator  n_maps  median_signed_err  median_abs_err  max_abs_err  slope  pass_preregistered  PASS
   b1p1_t900 all pixels       3              +2.30           +2.30        +5.50  +0.74                True  True
   b1p1_t900      low-S       3              -1.10           +2.30        +7.30  +0.79                True  True
 b1p.01_t900 all pixels       3              -3.20           +3.20        +8.30  +0.75                True  True
 b1p.01_t900      low-S       3              -4.40           +4.40        +9.80  +0.70                True  True
  b.1p1_t900 all pixels       3              +8.60           +8.60       +21.60  +0.02                True False
  b.1p1_t900      low-S       3              +1.00          +12.10       +17.60  +0.01                True False
b.1p.01_t750 all pixels       3              +7.90           +7.90       +23.10  +0.08                True False
b.1p.01_t750      low-S       3              +3.90          +13.80       +16.70  -0.02          

In [10]:
import shutil
shutil.copy("results_rotated.csv", "stageA_results.csv")
res = __import__("evaluate").grade_csv("results_rotated.csv")
res.to_csv("stageA_grades.csv", index=False)

In [11]:
def load(pattern):
    return fits.getdata(glob.glob(pattern)[0]).astype(float)

def pressure(run):            # "b1p.01" -> 0.01, "b.1p1" -> 1.0
    return float(run.split("p", 1)[1].replace(".", "0.", 1) if run.split("p", 1)[1].startswith(".") else run.split("p", 1)[1])

readme = {"b1p1": (0.7, 0.7), "b1p.01": (7, 0.7), "b.1p1": (0.7, 2.0), "b.1p.01": (7, 2.0)}

for run, (ms_rm, ma_rm) in readme.items():
    rho = load(f"{run}/dens_*")
    v = np.sqrt(sum(load(f"{run}/vel{k}_*")**2 for k in "xyz"))
    Bm = np.sqrt(sum(load(f"{run}/mag{k}_*")**2 for k in "xyz"))
    cs = np.sqrt(pressure(run))
    vA = Bm / np.sqrt(rho)
    print(f"{run:8s} cs={cs:.2f}  M_s={np.mean(v/cs):5.2f} (README ~{ms_rm})  "
          f"M_A={np.mean(v/vA):5.2f} (README ~{ma_rm})")

b1p1     cs=1.00  M_s= 0.74 (README ~0.7)  M_A= 0.73 (README ~0.7)
b1p.01   cs=0.10  M_s= 7.29 (README ~7)  M_A= 0.55 (README ~0.7)
b.1p1    cs=1.00  M_s= 0.75 (README ~0.7)  M_A= 2.26 (README ~2.0)
b.1p.01  cs=0.10  M_s= 7.11 (README ~7)  M_A= 1.92 (README ~2.0)


In [12]:
for run in ["b1p1", "b1p.01", "b.1p1", "b.1p.01"]:
    rho = load(f"{run}/dens_*")
    B = {k: load(f"{run}/mag{k}_*") for k in "xyz"}
    B0 = abs(B["x"].mean())
    dB = np.sqrt(sum(((B[k] - B[k].mean())**2).mean() for k in "xyz"))
    print(f"{run:8s} turbulent/mean field dB/B0 = {dB/B0:5.2f}   "
          f"density spread sigma(rho)/<rho> = {rho.std()/rho.mean():.2f}")

b1p1     turbulent/mean field dB/B0 =  0.58   density spread sigma(rho)/<rho> = 0.31
b1p.01   turbulent/mean field dB/B0 =  0.51   density spread sigma(rho)/<rho> = 1.94
b.1p1    turbulent/mean field dB/B0 =  5.80   density spread sigma(rho)/<rho> = 0.25
b.1p.01  turbulent/mean field dB/B0 =  4.44   density spread sigma(rho)/<rho> = 1.59


# AREPO SO THIS ONE IS WITH GRAVITY

In [13]:
for f in ["arepo_B100/Density_B100snap_156_512.hdf5",
          "arepo_B100/MagneticField_B100snap_156_512.hdf5"]:
    print(f)
    with h5py.File(f, "r") as h:
        h.visititems(lambda name, obj: print("  ", name, getattr(obj, "shape", ""),
                                             getattr(obj, "dtype", "")))
        for k, v in h.attrs.items():
            print("   attr", k, "=", v)

arepo_B100/Density_B100snap_156_512.hdf5
   Density (512, 512, 512) float64
arepo_B100/MagneticField_B100snap_156_512.hdf5
   MagneticField_0 (512, 512, 512) float64
   MagneticField_1 (512, 512, 512) float64
   MagneticField_2 (512, 512, 512) float64


In [14]:
%run prepare_arepo.py --dens arepo_B100/Density_B100snap_156_512.hdf5 --mag arepo_B100/MagneticField_B100snap_156_512.hdf5 --out arepo_B100/grid256

density: (256, 256, 256)  (1 s)
field:   3 x (256, 256, 256)  (3 s)

mean |div B| for each pairing (component 0,1,2 -> array axis):
   (0, 1, 2)  6.275e-04   <- chosen
   (1, 0, 2)  1.564e-03
   (2, 1, 0)  1.581e-03
   (2, 0, 1)  2.073e-03
   (1, 2, 0)  2.522e-03
   (0, 2, 1)  2.667e-03
best is 2.5x smaller than the next pairing

component labels: {'MagneticField_0': 'z', 'MagneticField_1': 'y', 'MagneticField_2': 'x'}
mean field:  <Bx> = -0.0002103  <By> = +7.968e-05  <Bz> = +0.1962
rms field:   rms Bx = 0.01376  rms By = 0.01343  rms Bz = 0.0122
density: mean 0.002823  max/mean 1.61e+03  sigma/mean 2.33

wrote dens/magx/magy/magz.fits to arepo_B100/grid256  (4 s total)


In [15]:
%run rotate_maps.py --rho arepo_B100/grid256/dens.fits --bx arepo_B100/grid256/magx.fits --by arepo_B100/grid256/magy.fits --bz arepo_B100/grid256/magz.fits --run AREPO_B100_snap156

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: AREPO_B100_snap156   shape: (256, 256, 256)   p_max rule: 99.5th percentile
mean field: <Bx>=-0.0002 <By>=+0.0001 <Bz>=+0.1962  -> field along z; tilting toward y

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.0    3.7  0.1573  0.1495   0.23    5.0    4.2   +5.0    +4.2     5.8      5.2   3.3
   15   15.0   15.0  0.1504  0.1432   0.25   11.1   10.9   -3.9    -4.1    16.0     15.9   3.3
   30   30.0   30.0  0.1267  0.1216   0.30   18.6   18.9  -11.4   -11.1    30.5     30.7   3.3
   45   45.0   44.7  0.0899  0.0873   0.42   25.2   24.8  -19.8   -20.2    45.2     44.9   3.3
   60   60.0   59.3  0.0508  0.0499   0.68   32.3   31.9  -27.7   -28.1    59.7     59.6   3.3
   75   75.0   74.3  0.0207  0.0206   1.61   48.8   48.3  -26.2   -26.7    75.2     75.0   3.2
   90   89.9   85.4  0.0073  0.

In [16]:
P0 = fm.P0_TRUE
mapping = {"x": 0, "y": 1, "z": 2}   # EDIT from prepare_arepo output: which MagneticField_i is x, y, z
sky = ...                            # EDIT: see note below

with h5py.File("arepo_B100/Density_B100snap_156_512.hdf5") as hd, \
     h5py.File("arepo_B100/MagneticField_B100snap_156_512.hdf5") as hb:
    n = hd["Density"].shape[0]
    N = Q = U = N2 = W = 0
    for i in range(0, n, 16):                           # sum along array axis 0
        r = hd["Density"][i:i+16]
        b = {k: hb[f"MagneticField_{j}"][i:i+16] for k, j in mapping.items()}
        b1, b2, bl = b["x"], b["y"], b["z"]             # assumes axis 0 is z (check!)
        bt = b1**2 + b2**2 + bl**2; c2 = (b1**2 + b2**2) / bt
        N  = N  + r.sum(0);  Q = Q + (r*(b2**2-b1**2)/bt).sum(0)
        U  = U  + (r*2*b1*b2/bt).sum(0); N2 = N2 + (r*(c2-2/3)).sum(0); W = W + (r*c2).sum(0)
p = P0*np.sqrt(Q**2+U**2)/(N-P0*N2)

In [17]:
%run check_arepo.py --dens arepo_B100/Density_B100snap_156_512.hdf5 --mag arepo_B100/MagneticField_B100snap_156_512.hdf5 --grid arepo_B100/grid256 --map z,y,x

ANISOTROPY CHECK (smaller = smoother along that direction)
  mean field is labelled along z
  along z (array axis 0):  log-density 0.0514   field fluctuations 1.068e-03
  along y (array axis 1):  log-density 0.1778   field fluctuations 3.932e-03
  along x (array axis 2):  log-density 0.2050   field fluctuations 4.437e-03
  smoothest direction: density z, field z  ->  CONSISTENT with field along z

RESOLUTION CHECK: full 512^3 vs block-averaged 256^3, unrotated views
            view  res   true   p_max    <S>  g_obs   lowS    err errLowS
  down z (along B)  512   89.9  0.0077   3.73   74.3   72.3  -15.6   -17.6
  down z (along B)  256   89.9  0.0073   8.01   73.7   71.5  -16.2   -18.4
  down y (across B)  512    0.0  0.1573   0.12    5.1    4.2   +5.1    +4.2
  down y (across B)  256    0.0  0.1573   0.23    5.0    4.2   +5.0    +4.2
  largest 512-vs-256 difference in the estimate: 0.8 deg  ->  averaging to 256^3 is fine


In [18]:
%run rotate_maps.py --rho arepo_B100/grid256/dens.fits --bx arepo_B100/grid256/magx.fits --by arepo_B100/grid256/magy.fits --bz arepo_B100/grid256/magz.fits --run AREPO_B100_snap156

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: AREPO_B100_snap156   shape: (256, 256, 256)   p_max rule: 99.5th percentile
mean field: <Bx>=-0.0002 <By>=+0.0001 <Bz>=+0.1962  -> field along z; tilting toward y

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.0    3.7  0.1573  0.1495   0.23    5.0    4.2   +5.0    +4.2     5.8      5.2   3.2
   15   15.0   15.0  0.1504  0.1432   0.25   11.1   10.9   -3.9    -4.1    16.0     15.9   3.2
   30   30.0   30.0  0.1267  0.1216   0.30   18.6   18.9  -11.4   -11.1    30.5     30.7   3.3
   45   45.0   44.7  0.0899  0.0873   0.42   25.2   24.8  -19.8   -20.2    45.2     44.9   3.2
   60   60.0   59.3  0.0508  0.0499   0.68   32.3   31.9  -27.7   -28.1    59.7     59.6   3.2
   75   75.0   74.3  0.0207  0.0206   1.61   48.8   48.3  -26.2   -26.7    75.2     75.0   3.2
   90   89.9   85.4  0.0073  0.

In [19]:
%run prepare_arepo.py --dens arepo_B30/Density_B30snap_073_512.hdf5 --mag arepo_B30/MagneticField_B30snap_073_512.hdf5 --out arepo_B30/grid256

density: (256, 256, 256)  (1 s)
field:   3 x (256, 256, 256)  (3 s)

mean |div B| for each pairing (component 0,1,2 -> array axis):
   (0, 1, 2)  9.548e-04   <- chosen
   (2, 1, 0)  2.685e-03
   (1, 0, 2)  2.689e-03
   (0, 2, 1)  3.319e-03
   (2, 0, 1)  3.386e-03
   (1, 2, 0)  3.498e-03
best is 2.8x smaller than the next pairing

component labels: {'MagneticField_0': 'z', 'MagneticField_1': 'y', 'MagneticField_2': 'x'}
mean field:  <Bx> = +0.0002619  <By> = -0.0004013  <Bz> = +0.0584
rms field:   rms Bx = 0.02296  rms By = 0.02184  rms Bz = 0.0197
density: mean 0.003002  max/mean 1.71e+03  sigma/mean 2.48

wrote dens/magx/magy/magz.fits to arepo_B30/grid256  (4 s total)


In [20]:
%run check_arepo.py --dens arepo_B30/Density_B30snap_073_512.hdf5 --mag arepo_B30/MagneticField_B30snap_073_512.hdf5 --grid arepo_B30/grid256 --map z,y,x

ANISOTROPY CHECK (smaller = smoother along that direction)
  mean field is labelled along z
  along z (array axis 0):  log-density 0.1268   field fluctuations 3.771e-03
  along y (array axis 1):  log-density 0.1943   field fluctuations 6.151e-03
  along x (array axis 2):  log-density 0.1928   field fluctuations 6.172e-03
  smoothest direction: density z, field z  ->  CONSISTENT with field along z

RESOLUTION CHECK: full 512^3 vs block-averaged 256^3, unrotated views
            view  res   true   p_max    <S>  g_obs   lowS    err errLowS
  down z (along B)  512   89.5  0.0603   3.52   60.6   57.0  -28.9   -32.5
  down z (along B)  256   89.5  0.0602   7.06   62.9   55.7  -26.6   -33.8
  down y (across B)  512    0.4  0.1418   0.94   28.7   25.0  +28.3   +24.6
  down y (across B)  256    0.4  0.1419   1.81   29.5   25.2  +29.1   +24.8
  largest 512-vs-256 difference in the estimate: 2.3 deg  ->  averaging to 256^3 is fine


In [21]:
%run rotate_maps.py --rho arepo_B30/grid256/dens.fits --bx arepo_B30/grid256/magx.fits --by arepo_B30/grid256/magy.fits --bz arepo_B30/grid256/magz.fits --run AREPO_B30_snap073

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: AREPO_B30_snap073   shape: (256, 256, 256)   p_max rule: 99.5th percentile
mean field: <Bx>=+0.0003 <By>=-0.0004 <Bz>=+0.0584  -> field along z; tilting toward y

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.4   16.8  0.1419  0.1355   1.80   29.5   25.1  +29.1   +24.7    33.7     30.0   3.3
   15   14.6   20.5  0.1386  0.1325   1.85   29.4   24.6  +14.8   +10.0    34.0     29.9   3.4
   30   29.6   29.2  0.1298  0.1244   2.13   39.1   33.4   +9.5    +3.8    43.2     39.6   3.3
   45   44.6   40.0  0.1118  0.1078   2.94   46.7   43.6   +2.1    -1.0    53.4     51.2   3.5
   60   59.6   53.2  0.0866  0.0842   4.42   57.4   49.3   -2.2   -10.3    64.9     59.8   3.4
   75   74.6   59.6  0.0608  0.0596   6.49   57.8   54.5  -16.8   -20.1    69.2     67.0   3.1
   90   89.5   61.0  0.0602  0.0

In [22]:
%run sensitivity.py --rho b1p1/dens_t900.fits.gz --bx b1p1/magx_t900.fits.gz --by b1p1/magy_t900.fits.gz --bz b1p1/magz_t900.fits.gz --run b1p1_t900

run: b1p1_t900   field along x
estimate (deg) for each p_max rule and peak method; err = estimate - true

angle  true p_max rule      p0    KDE    err reflKDE    err median    err
    0   0.0        max  0.1388   26.5  +26.5    26.5  +26.5   28.7  +28.7
    0   0.0      p99.9  0.1371   25.8  +25.8    25.8  +25.8   28.1  +28.1
    0   0.0      p99.5  0.1348   25.0  +25.0    25.0  +25.0   27.3  +27.3
    0   0.0        p99  0.1325   26.9  +26.9    26.9  +26.9   26.4  +26.4
    0   0.0   p0 known  0.1500   30.1  +30.1    30.1  +30.1   32.2  +32.2

   30  30.0        max  0.1143   33.9   +3.9    33.9   +3.9   34.5   +4.5
   30  30.0      p99.9  0.1111   33.0   +3.0    33.0   +3.0   33.4   +3.4
   30  30.0      p99.5  0.1093   32.3   +2.3    32.3   +2.3   32.7   +2.7
   30  30.0        p99  0.1082   31.9   +1.9    31.9   +1.9   32.3   +2.3
   30  30.0   p0 known  0.1500   41.7  +11.7    41.7  +11.7   43.3  +13.3

   60  60.0        max  0.0873   56.7   -3.3    56.7   -3.3   56.6   -3.4
   6

In [23]:
%run sensitivity.py --rho arepo_B100/grid256/dens.fits --bx arepo_B100/grid256/magx.fits --by arepo_B100/grid256/magy.fits --bz arepo_B100/grid256/magz.fits --run AREPO_B100_snap156

run: AREPO_B100_snap156   field along z
estimate (deg) for each p_max rule and peak method; err = estimate - true

angle  true p_max rule      p0    KDE    err reflKDE    err median    err
    0   0.0        max  0.1496    5.2   +5.2     5.2   +5.2    5.8   +5.7
    0   0.0      p99.9  0.1496    5.1   +5.1     5.1   +5.1    5.6   +5.6
    0   0.0      p99.5  0.1495    5.0   +5.0     5.0   +5.0    5.5   +5.4
    0   0.0        p99  0.1494    4.9   +4.9     4.9   +4.9    5.4   +5.3
    0   0.0   p0 known  0.1500    5.8   +5.8     5.8   +5.8    6.3   +6.3

   30  30.0        max  0.1321   24.1   -5.9    24.1   -5.9   24.0   -6.0
   30  30.0      p99.9  0.1259   21.1   -8.9    21.1   -8.9   20.9   -9.1
   30  30.0      p99.5  0.1216   18.6  -11.4    18.6  -11.4   18.4  -11.7
   30  30.0        p99  0.1198   17.4  -12.6    17.4  -12.6   17.1  -12.9
   30  30.0   p0 known  0.1500   30.5   +0.5    30.5   +0.5   30.5   +0.4

   60  60.0        max  0.0752   45.9  -14.1    45.9  -14.1   46.4  -

In [24]:
cho = [("b1p1", "t900", ".gz"), ("b1p.01", "t900", ".gz"),
       ("b.1p1", "t900", ".gz"), ("b.1p.01", "t750", "")]
for run, t, gz in cho:
    %run rotate_maps.py --rho {run}/dens_{t}.fits{gz} --bx {run}/magx_{t}.fits.gz --by {run}/magy_{t}.fits.gz --bz {run}/magz_{t}.fits.gz --run {run}_{t} --pmax-percentile 100

for B, snap in [("B100", "156"), ("B30", "073")]:
    g = f"arepo_{B}/grid256"
    %run rotate_maps.py --rho {g}/dens.fits --bx {g}/magx.fits --by {g}/magy.fits --bz {g}/magz.fits --run AREPO_{B}_snap{snap} --pmax-percentile 100

[check] uniform field at 0/30/45/60/90 deg matches Chen eq. 7   OK
[check] 0 deg and 90 deg reproduce the unrotated z and x views  OK

run: b1p1_t900_pmax100   shape: (256, 256, 256)   p_max rule: map maximum
mean field: <Bx>=+1.0000 <By>=+0.0000 <Bz>=-0.0000  -> field along x; tilting toward z

angle   true   g_2D   p_max  p0_der    <S>  g_obs   lowS    err errLowS p0known p0k_lowS   sec
    0    0.0   16.1  0.1455  0.1388   0.34   26.5   24.5  +26.5   +24.5    30.1     28.5   3.1
   15   15.0   26.5  0.1360  0.1301   0.39   33.3   27.9  +18.3   +12.9    38.3     34.0   3.6
   30   30.0   29.3  0.1188  0.1143   0.50   33.9   30.8   +3.9    +0.8    41.7     40.4   4.9
   45   45.0   47.6  0.1190  0.1145   0.80   51.8   43.1   +6.8    -1.9    56.8     49.7   6.0
   60   60.0   53.6  0.0899  0.0873   1.56   56.7   55.2   -3.3    -4.8    64.6     63.6   6.9
   75   75.0   63.0  0.0750  0.0732   2.66   66.5   62.5   -8.5   -12.5    72.9     70.7   7.0
   90   90.0   67.2  0.0652  0.0638   

In [25]:
%run evaluate.py

                       run  estimator  n_maps  median_signed_err  median_abs_err  max_abs_err  slope  pass_preregistered  PASS
                 b1p1_t900 all pixels       3              +2.30           +2.30        +5.50  +0.74                True  True
                 b1p1_t900      low-S       3              -1.10           +2.30        +7.30  +0.79                True  True
               b1p.01_t900 all pixels       3              -3.20           +3.20        +8.30  +0.75                True  True
               b1p.01_t900      low-S       3              -4.40           +4.40        +9.80  +0.70                True  True
                b.1p1_t900 all pixels       3              +8.60           +8.60       +21.60  +0.02                True False
                b.1p1_t900      low-S       3              +1.00          +12.10       +17.60  +0.01                True False
              b.1p.01_t750 all pixels       3              +7.90           +7.90       +23.10  +0.08           

In [26]:
%run measure_turbulence.py --freeze stageAB

wrote stageAB_results.csv and stageAB_grades.csv


In [27]:
%run monte_carlo.py

check: analytic eq. 7 vs Table 1 analytic column
  cos^2 gamma = 0.9:  0.092  (Table 1: 0.092)
  cos^2 gamma = 0.7:  0.070  (Table 1: 0.070)
  cos^2 gamma = 0.5:  0.049  (Table 1: 0.049)
  cos^2 gamma = 0.3:  0.029  (Table 1: 0.029)
  cos^2 gamma = 0.1:  0.009  (Table 1: 0.009)

Monte Carlo peak p (M = 50, N = 100000, p0 = 0.1, doubled (axial) angles, KDE on log p); Chen Table 1 in brackets
 cos2        kg=16,kp=16         kg=1,kp=16         kg=16,kp=1          kg=1,kp=1
  0.9      0.088 [0.088]      0.066 [0.068]      0.043 [0.043]      0.033 [0.038]
  0.7      0.067 [0.068]      0.057 [0.061]      0.033 [0.034]      0.028 [0.030]
  0.5      0.048 [0.049]      0.048 [0.049]      0.023 [0.024]      0.024 [0.024]
  0.3      0.029 [0.029]      0.039 [0.037]      0.014 [0.014]      0.019 [0.018]
  0.1      0.010 [0.010]      0.030 [0.031]      0.005 [0.005]      0.015 [0.015]

largest difference from Table 1: 0.005  (check the larger differences)
